In [1]:
import requests 
import pandas as pd  
import numpy as np  
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_squared_error, accuracy_score
from datetime import datetime, timedelta 
import pytz
import joblib
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

api_key = 'd881de71e6cf19a923c0139398067320'
base_url = 'https://api.openweathermap.org/data/2.5/weather'

def get_current_weather(city):
    url = f"{base_url}?q={city}&appid={api_key}&units=metric"
    response = requests.get(url)
    data = response.json()

    if response.status_code != 200:
        print(f"[ERROR] Could not retrieve data for {city}")
        print(f"[DETAILS] {data}")
        return None

    return {
        'city': data['name'],
        'current_temp': round(data['main']['temp']),
        'feels_like': round(data['main']['feels_like']),
        'temp_min': round(data['main']['temp_min']),
        'temp_max': round(data['main']['temp_max']),
        'humidity': data['main']['humidity'],
        'description': data['weather'][0]['description'],
        'country': data['sys']['country'],
        'wind_gust_dir': data['wind'].get('deg', 0),
        'pressure': data['main']['pressure'],
        'wind_Gust_Speed': data['wind']['speed']
    }

In [2]:
def file_reading(filename):
    df = pd.read_csv(filename)
    df = df.dropna()
    df = df.drop_duplicates()
    return df

def prepare_data(data):
    label = LabelEncoder()
    data['WindGustDir'] = label.fit_transform(data['WindGustDir'])
    data['RainTomorrow'] = label.fit_transform(data['RainTomorrow'])
    
    # Feature engineering - add rolling averages
    data['3day_avg_temp'] = data['Temp'].rolling(3).mean()
    data['prev_humidity'] = data['Humidity'].shift(1)
    
    X = data[['MinTemp', 'MaxTemp', 'WindGustDir', 'WindGustSpeed', 
              'Humidity', 'Pressure', 'Temp', '3day_avg_temp', 'prev_humidity']]
    y = data['RainTomorrow']

    return X, y, label

def create_sequences(data, n_steps=3):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps])
        y.append(data[i+n_steps])
    return np.array(X), np.array(y)

In [3]:
def train_rain_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Handle class imbalance
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
    
    model = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    return model

In [4]:
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def train_lstm_model(data, feature, n_steps=3):
    # Scale data (reshape to 2D for scaling, then back to original shape)
    scaler = MinMaxScaler()
    values = data[feature].values.reshape(-1, 1)
    scaled_data = scaler.fit_transform(values).flatten()
    
    # Create sequences
    X, y = create_sequences(scaled_data, n_steps)
    
    # Split data
    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    
    # Build and train model
    model = build_lstm_model((n_steps, 1))
    early_stop = EarlyStopping(monitor='val_loss', patience=5)
    model.fit(X_train, y_train, 
              epochs=50, 
              batch_size=32, 
              validation_data=(X_test, y_test),
              callbacks=[early_stop],
              verbose=0)
    
    return model, scaler

In [5]:
def predict_future_lstm(model, scaler, current_values, n_steps=3, n_future=5):
    predictions = []
    # Scale the current values (reshape to 2D for scaling)
    scaled_values = scaler.transform(np.array(current_values).reshape(-1, 1)).flatten()
    current_sequence = scaled_values[-n_steps:].reshape(1, n_steps, 1)
    
    for _ in range(n_future):
        next_pred = model.predict(current_sequence, verbose=0)
        predictions.append(scaler.inverse_transform(next_pred)[0][0])
        # Update sequence (remove oldest, add new prediction)
        current_sequence = np.append(current_sequence[:, 1:, :], 
                                   next_pred.reshape(1, 1, 1), 
                                   axis=1)
    
    return predictions

In [6]:
def weather_view():
    city = input("Enter any city name: ")
    current_weather = get_current_weather(city)

    if current_weather is None:
        print("[ERROR] Could not retrieve weather data for the city.")
        return

    data = file_reading('weather.csv')
    X, y, label_encoder = prepare_data(data)
    rain_model = train_rain_model(X, y)

    # Wind direction processing
    wind_deg = current_weather.get('wind_gust_dir', 0) % 360
    compass_points = [
        ("N", 0, 11.25), ("NNE", 11.25, 33.75), ("NE", 33.75, 56.25),
        ("ENE", 56.25, 78.75), ("E", 78.75, 101.25), ("ESE", 101.25, 123.75),
        ("SE", 123.75, 146.25), ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
        ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25), ("WSW", 236.25, 258.75),
        ("W", 258.75, 281.25), ("WNW", 281.25, 303.75), ("NW", 303.75, 326.25),
        ("NNW", 326.25, 348.75), ("N", 348.75, 360)
    ]
    compass_direction = next(
        (point for point, start, end in compass_points if start <= wind_deg < end), "N"
    )

    # Encode wind direction
    if compass_direction in label_encoder.classes_:
        encoded_wind_dir = label_encoder.transform([compass_direction])[0]
    else:
        encoded_wind_dir = -1

    # Prepare current data for rain prediction
    current_df = pd.DataFrame([{
        'MinTemp': current_weather['temp_min'],
        'MaxTemp': current_weather['temp_max'],
        'WindGustDir': encoded_wind_dir,
        'WindGustSpeed': current_weather['wind_Gust_Speed'],
        'Humidity': current_weather['humidity'],
        'Pressure': current_weather['pressure'],
        'Temp': current_weather['current_temp'],
        '3day_avg_temp': current_weather['current_temp'],  # Simplified for demo
        'prev_humidity': current_weather['humidity']  # Simplified for demo
    }])

    # Predict rain
    rain_prediction = rain_model.predict(current_df)[0]

    # Train LSTM models
    temp_model, temp_scaler = train_lstm_model(data, 'Temp')
    hum_model, hum_scaler = train_lstm_model(data, 'Humidity')

    # Get recent values for LSTM prediction
    recent_temp = data['Temp'].tail(3).values
    recent_hum = data['Humidity'].tail(3).values

    # Predict future values
    future_temp = predict_future_lstm(temp_model, temp_scaler, recent_temp)
    future_humidity = predict_future_lstm(hum_model, hum_scaler, recent_hum)

    # Time labels
    timezone = pytz.timezone('Asia/Kolkata')
    now = datetime.now(timezone)
    future_times = [(now + timedelta(hours=i+1)).strftime('%H:00') for i in range(5)]

    # Save models
    joblib.dump(rain_model, 'rain_model_xgb.joblib')
    temp_model.save('temp_model_lstm.h5')
    hum_model.save('humidity_model_lstm.h5')

    # Display results
    print(f"\nCity: {city}, {current_weather['country']}")
    print(f"Current Temperature: {current_weather['current_temp']}°C")
    print(f"Feels Like: {current_weather['feels_like']}°C")
    print(f"Humidity: {current_weather['humidity']}%")
    print(f"Weather: {current_weather['description']}")
    print(f"Rain Prediction: {'Yes' if rain_prediction else 'No'}")

    print("\nFuture Temperature Predictions (LSTM):")
    for time, temp in zip(future_times, future_temp):
        print(f"{time}: {round(temp, 1)}°C")

    print("\nFuture Humidity Predictions (LSTM):")
    for time, hum in zip(future_times, future_humidity):
        print(f"{time}: {round(hum, 1)}%")

    # Recommendations
    if rain_prediction == 1:
        print("🌧️ It might rain tomorrow. No need to irrigate.")
    elif current_weather['humidity'] < 60:
        print("💧 It's dry. Consider irrigating your crops.")
    else:
        print("✅ No rain expected and humidity is decent.")

    if current_weather['temp_max'] > 35:
        print("🌡️ It's hot. Harvest early morning to avoid heat stress.")
        
    if current_weather['wind_Gust_Speed'] > 30:
        print("🌬️ Strong winds. Avoid spraying pesticides or fertilizers today.")

In [7]:
weather_view()

Accuracy: 0.88

City: Chennai, IN
Current Temperature: 29°C
Feels Like: 35°C
Humidity: 84%
Weather: scattered clouds
Rain Prediction: Yes

Future Temperature Predictions (LSTM):
10:00: 21.799999237060547°C
11:00: 24.299999237060547°C
12:00: 23.799999237060547°C
13:00: 21.899999618530273°C
14:00: 22.5°C

Future Humidity Predictions (LSTM):
10:00: 42.900001525878906%
11:00: 39.599998474121094%
12:00: 38.400001525878906%
13:00: 44.29999923706055%
14:00: 43.900001525878906%
🌧️ It might rain tomorrow. No need to irrigate.


c:\Users\Kumaran N\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
